Importy

In [ ]:
import os
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array

Konfigurace

In [ ]:
IMG_SIZE = 64                       # Velikost obrázku, na kterou se škálují vstupy
PATCH_SIZE = 32                     # Velikost patchů pro model (rozsah 32x32)
CLASS_NAMES = ["L-mode", "H-mode", "ELM"]  # Seznam tříd
MODEL_PATH = "complex_model.h5"     # Cesta k uloženému modelu


Načtení modelu

In [ ]:
try:
    model = load_model(MODEL_PATH)
    print(f"Model úspěšně načten z: {MODEL_PATH}")
except Exception as e:
    print(f"Chyba při načítání modelu: {e}")
    exit()

Předzpracování obrázku

In [ ]:
def preprocess_image(image_path):
    """
    Načte obrázek, škáluje ho, rozdělí na PATCH_SIZE x PATCH_SIZE patchy.
    Vrací numpy pole patchů.
    """
    # Načtení obrázku s 4 kanály (RGBA)
    img = load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE), color_mode='rgba')
    img_array = img_to_array(img) / 255.0  # normalizace pixelů 0..1

    patches = []
    # Rozdělení obrázku na menší čtverce (patches)
    for i in range(0, IMG_SIZE, PATCH_SIZE):
        for j in range(0, IMG_SIZE, PATCH_SIZE):
            patch = img_array[i:i+PATCH_SIZE, j:j+PATCH_SIZE]
            if patch.shape == (PATCH_SIZE, PATCH_SIZE, 4):  # kontrola velikosti
                patches.append(patch)
    return np.array(patches)

Predikce pro jeden obrázek

In [ ]:
def predict_image(model, image_path):
    """
    Spustí model na všech patchech obrázku a vrátí průměrné pravděpodobnosti,
    predikovanou třídu a jistotu.
    """
    if not os.path.exists(image_path):
        print(f"Obrázek nebyl nalezen: {image_path}")
        return None, None, None

    patches = preprocess_image(image_path)
    if patches.size == 0:
        print(f"Obrázek {image_path} neobsahuje validní patchy.")
        return None, None, None

    predictions = model.predict(patches)                 # predikce pro každý patch
    avg_prediction = np.mean(predictions, axis=0)        # průměr přes všechny patchy
    predicted_class = np.argmax(avg_prediction)          # index třídy s max pravděpodobností
    predicted_label = CLASS_NAMES[predicted_class]      # název třídy
    confidence = float(avg_prediction[predicted_class]) # jistota modelu

    return predicted_label, confidence, avg_prediction


Predikce na jednom obrázku (výpis)

In [ ]:
def predict_single_image(image_path):
    predicted_label, confidence, probabilities = predict_image(model, image_path)
    if predicted_label is None:
        return
    print(f"Obrázek: {os.path.basename(image_path)}")
    print(f"Predikovaná třída: {predicted_label}")
    print(f"Jistota: {confidence*100:.2f}%")
    print(f"Pravděpodobnosti: {probabilities}")

Predikce na složce obrázků

In [ ]:
def predict_images_in_folder(folder_path):
    if not os.path.exists(folder_path):
        print(f"Složka nebyla nalezena: {folder_path}")
        return

    for file in os.listdir(folder_path):
        if file.lower().endswith(('.png', '.jpg')):
            image_path = os.path.join(folder_path, file)
            predicted_label, confidence, probabilities = predict_image(model, image_path)
            if predicted_label is None:
                continue
            print(f"Obrázek: {file}")
            print(f"Predikovaná třída: {predicted_label}")
            print(f"Jistota: {confidence*100:.2f}%")
            print(f"Pravděpodobnosti: {probabilities}")
            print("-" * 50)

Hlavní část

In [ ]:
if __name__ == "__main__":
    # Predikce na jednom obrázku
    base_dir = os.path.dirname(os.path.abspath(__file__))
    single_image_path = os.path.join(base_dir, "data", "test.jpg")  # nahraď cestou k obrázku
    print("Predikce na jednom obrázku:")
    predict_single_image(single_image_path)
    print("\n" + "="*50 + "\n")

    # Predikce na složce obrázků
    test_folder = os.path.join(base_dir, "data", "img")  # nahraď cestou k složce
    print("Predikce na složce obrázků:")
    predict_images_in_folder(test_folder)